<h2>Final Project - Fake & Real News Detector</h2>

In [ ]:
!pip install accelerate -U

  Using cached accelerate-1.13.0-py3-none-any.whl.metadata (19 kB)
Using cached accelerate-1.13.0-py3-none-any.whl (383 kB)
  Attempting uninstall: accelerate
    Found existing installation: accelerate 0.30.1
    Uninstalling accelerate-0.30.1:
      Successfully uninstalled accelerate-0.30.1


In [ ]:
!pip uninstall -y transformers accelerate datasets peft

Found existing installation: transformers 4.41.2
Uninstalling transformers-4.41.2:
  Successfully uninstalled transformers-4.41.2
Found existing installation: accelerate 1.13.0
Uninstalling accelerate-1.13.0:
  Successfully uninstalled accelerate-1.13.0
Found existing installation: datasets 2.19.0
Uninstalling datasets-2.19.0:
  Successfully uninstalled datasets-2.19.0
Found existing installation: peft 0.18.1
Uninstalling peft-0.18.1:
  Successfully uninstalled peft-0.18.1


In [ ]:
!pip install transformers==4.41.2 datasets==2.19.0 accelerate==0.30.1 -q

In [3]:
# The modules we're going to use
from __future__ import print_function
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
import transformers
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import Trainer, TrainingArguments
from datasets import Dataset

import datasets
import accelerate

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("accelerate:", accelerate.__version__)

torch: 2.10.0+cu128
transformers: 5.0.0
datasets: 4.0.0
accelerate: 1.13.0


<h4>Task  1: Load & Split the Data</h4>

In [4]:
import kagglehub
path = kagglehub.dataset_download("clmentbisaillon/fake-and-real-news-dataset")

Using Colab cache for faster access to the 'fake-and-real-news-dataset' dataset.


In [5]:
# Load data from fake news dataset
path = "/kaggle/input/fake-and-real-news-dataset"
fake = pd.read_csv(os.path.join(path, "Fake.csv"))
real = pd.read_csv(os.path.join(path, "True.csv"))

fake["label"] = 0
real["label"] = 1

# Combine Both "real" & "fake" Sets and Randomize
df = pd.concat([fake, real]).sample(frac=1).reset_index(drop=True)

# Use only text
df = df[["text", "label"]]

# Test print
print(df.head())

# Split Data
texts_training, texts_validation, labels_training, labels_validation = train_test_split(
    df["text"], df["label"], test_size=0.2, random_state=42
)



                                                text  label
0  NEW YORK (Reuters) - Democratic front-runner H...      1
1  Today, the Presidential Inaugural Committee (P...      0
2  DANIEL GREENFIELD NAILS IT! This is one of the...      0
3  PRESIDENT TRUMP MADE A SURPRISE DETOUR ON HIS ...      0
4  21st Century Wire says Three weeks ago, while ...      0


<h4>Baseline Logistic Regression Model</h4>

In [ ]:
vectorizer = TfidfVectorizer(max_features=5000)

X_train = vectorizer.fit_transform(texts_training)
X_val = vectorizer.transform(texts_validation)

model = LogisticRegression()
model.fit(X_train, labels_training)

preds = model.predict(X_val)

print(classification_report(labels_validation, preds))

              precision    recall  f1-score   support

           0       0.99      0.99      0.99      4686
           1       0.99      0.99      0.99      4294

    accuracy                           0.99      8980
   macro avg       0.99      0.99      0.99      8980
weighted avg       0.99      0.99      0.99      8980



<h4>BERT Model - Loading & Processing Dataset</h4>

In [6]:
# Define Tokenizer
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize(batch):
    return tokenizer(batch["text"], padding=True, truncation=True)

# Define datasets
train_dataset = Dataset.from_dict({
    "text": texts_training.tolist(),
    "label": labels_training.tolist()
})

val_dataset = Dataset.from_dict({
    "text": texts_validation.tolist(),
    "label": labels_validation.tolist()
})

train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset = val_dataset.map(tokenize, batched=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/35918 [00:00<?, ? examples/s]

Map:   0%|          | 0/8980 [00:00<?, ? examples/s]

<h4>BERT Model - Training</h4>

In [ ]:
# Model definition
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=2
)

# Training setup
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

# Run Trainer
trainer.train()

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.12/dist-packages/transformers/training_args.py:1474: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existi

 3


wandb: You chose "Don't visualize my results"
wandb: Using W&B in offline mode.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss
1,0.000000,0.001395
2,0.000000,0.001940


TrainOutput(global_step=8980, training_loss=0.003871523052140788, metrics={'train_runtime': 4466.4139, 'train_samples_per_second': 16.084, 'train_steps_per_second': 2.011, 'total_flos': 9515928049852416.0, 'train_loss': 0.003871523052140788, 'epoch': 2.0})

<h4>BERT Model - Evaluation</h4>



In [ ]:
trainer.evaluate()

{'eval_loss': 0.0019404868362471461,
 'eval_runtime': 157.6913,
 'eval_samples_per_second': 56.947,
 'eval_steps_per_second': 7.122,
 'epoch': 2.0}

<h4>BERT Model - Saving the Model</h4>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

# Setup path
save_path = "/content/drive/MyDrive/fake-news-model"

Mounted at /content/drive


In [ ]:
trainer.save_model("/content/fake-news-model")
tokenizer.save_pretrained("/content/fake-news-model")

trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)

Mounted at /content/drive


('/content/drive/MyDrive/fake-news-model/tokenizer_config.json',
 '/content/drive/MyDrive/fake-news-model/special_tokens_map.json',
 '/content/drive/MyDrive/fake-news-model/vocab.txt',
 '/content/drive/MyDrive/fake-news-model/added_tokens.json',
 '/content/drive/MyDrive/fake-news-model/tokenizer.json')

<h4>BERT Model - Loading a Model</h4>

In [7]:
# Loading model and tokenizer from SavePath
model = AutoModelForSequenceClassification.from_pretrained(save_path)
tokenizer = AutoTokenizer.from_pretrained(save_path)

Loading weights:   0%|          | 0/104 [00:01<?, ?it/s]

In [8]:
# Testing Code Segment
text = "Breaking news: scientists discover cure for aging"

inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)

outputs = model(**inputs)
prediction = torch.argmax(outputs.logits)

print(prediction.item())

0


<h4>Project Demo - Collab App</h4>

In [9]:
!pip install streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 74.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 68.0 MB/s eta 0:00:00


In [13]:
!npm install -g localtunnel
!streamlit run app.py &>/content/logs.txt &
!lt --port 8501

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧
changed 22 packages in 5s
⠧
⠧3 packages are looking for funding
⠧  run `npm fund` for details
⠧your url is: https://fresh-bobcats-try.loca.lt
^C
